## Простой RAG без внешних моделей эмбеддингов

In [ ]:
import numpy as np
from collections import Counter
import re

# Источник знаний
knowledge_base = """
Машинное обучение - это раздел искусственного интеллекта.
Нейронные сети используются для распознавания образов.
Глубокое обучение является подмножеством машинного обучения.
Алгоритмы обучения с учителем требуют размеченных данных.
Кластеризация - это метод обучения без учителя.
"""

# Разбиваем на фрагменты
def split_text(text):
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    return sentences

fragments = split_text(knowledge_base)

# Простая векторизация на основе TF-IDF вручную
def create_vocabulary(texts):
    """Создаём словарь всех уникальных слов"""
    all_words = []
    for text in texts:
        words = re.findall(r'\w+', text.lower())
        all_words.extend(words)
    return sorted(set(all_words))

def text_to_vector(text, vocabulary):
    """Преобразуем текст в вектор на основе частоты слов"""
    words = re.findall(r'\w+', text.lower())
    word_count = Counter(words)
    
    vector = []
    for word in vocabulary:
        vector.append(word_count.get(word, 0))
    
    return np.array(vector, dtype=float)

def cosine_similarity(vec1, vec2):
    """Вычисляем косинусное сходство между векторами"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        return 0
    
    return dot_product / (norm1 * norm2)

# Подготавливаем данные
vocabulary = create_vocabulary(fragments)
fragment_vectors = [text_to_vector(frag, vocabulary) for frag in fragments]


# RAG функция
def simple_rag(query, top_k=2):
    """Поиск наиболее релевантных фрагментов"""
    query_vector = text_to_vector(query, vocabulary)
    
    # Вычисляем схожесть с каждым фрагментом
    similarities = []
    for i, frag_vec in enumerate(fragment_vectors):
        sim = cosine_similarity(query_vector, frag_vec)
        similarities.append((i, sim))
    
    # Сортируем по убыванию схожести
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Возвращаем топ-k фрагментов
    results = []
    for i, sim in similarities[:top_k]:
        if sim > 0:  # Только релевантные результаты
            results.append({
                'fragment': fragments[i],
                'similarity': sim
            })
    
    return results


# Пример использования
query = "Что такое глубокое обучение?"
results = simple_rag(query, top_k=2)

print(f"Запрос: {query}\n")
for i, result in enumerate(results, 1):
    print(f"Результат {i} (релевантность: {result['similarity']:.3f}):")
    print(f"{result['fragment']}\n")

Запрос: Что такое глубокое обучение?

Результат 1 (релевантность: 0.577):
Глубокое обучение является подмножеством машинного обучения

Результат 2 (релевантность: 0.289):
Машинное обучение - это раздел искусственного интеллекта



## Как измеряют качество RAG на практике

В базовых проектах обычно ограничиваются сравнением precision/recall или субъективной проверкой достоверности ответов

### Как применять Precision и Recall в RAG

Предположим, у нас есть агент, который по запросу пользователя извлекает несколько документов (или фрагментов текста) из базы.
Мы знаем, какие документы должны быть найдены (это “истинные релевантные” документы), и можем сравнить результаты поиска с эталоном.

In [2]:
# Истинно релевантные документы
true_docs = {"doc1", "doc3", "doc5"}

# Документы, извлечённые агентом RAG
retrieved_docs = {"doc2", "doc3", "doc5", "doc6"}

# Вычисляем пересечение
true_positives = len(true_docs & retrieved_docs)

# Precision и Recall
precision = true_positives / len(retrieved_docs)
recall = true_positives / len(true_docs)

# F1-метрика
if precision + recall > 0:
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    f1 = 0.0

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

Precision: 0.50
Recall: 0.67
F1 Score: 0.57


В продвинутых системах применяются метрики, оценивающие **насколько ответ модели соответствует фактам**, а не только найденным документам. Эти методы позволяют глубже оценить качество работы RAG-агентов, но требуют вычислительных ресурсов и дополнительных данных. Тут мы не будем останавливаться на них подробно, но вот некоторые из подходов:
- **Faithfulness / Factual consistency** — проверка, не «галлюцинирует» ли модель, то есть соответствует ли ответ извлечённым данным
- **Context precision / recall** — насколько хорошо подобран контекст для генерации
- **Answer relevance** — насколько ответ полезен и отвечает на вопрос пользователя
- **RAGAS** — популярная библиотека от Hugging Face для комплексной оценки RAG-агентов
- **NLI-модели (Natural Language Inference)** — проверяют, подтверждает ли контекст утверждения из ответа